# IPSA (CC01-1940): EDA y preprocesamiento

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanestebancg2806/sugarcane-yield-quality-prediction/blob/main/01_eda_ipsa.ipynb)

Este notebook cubre el inventario, la calidad de datos y el EDA sobre `BD_IPSA_1940.xlsx`: el recorte de la variedad **CC01-1940** del Ingenio Providencia. Es un estudio distinto del histórico de suertes (`01_eda_suertes.ipynb` / `02_modelos_regresion.ipynb`).

Cada fila es un registro IPSA de esa variedad. El flujo es inventario y calidad de datos, EDA y selección de predictores. Los modelos de clasificación (logística multinomial y KNN) están en `02_modelos_clasificacion.ipynb`.

En Colab hay que subir `BD_IPSA_1940.xlsx` al directorio de trabajo (el Excel no está en el repositorio).


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats



pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

%matplotlib inline

In [2]:
# Lectura del dataset IPSA
df_ipsa = pd.read_excel("BD_IPSA_1940.xlsx", sheet_name="BD_IPSA")

# La primera columna de IPSA es un índice residual del Excel
if "Unnamed: 0" in df_ipsa.columns:
    df_ipsa = df_ipsa.drop(columns=["Unnamed: 0"])

print("BD_IPSA:", df_ipsa.shape)


BD_IPSA: (2187, 20)


In [3]:
# Vista rápida: IPSA (variedad CC01-1940)
display(df_ipsa.head())
display(df_ipsa.info())
display(df_ipsa.describe(include="all").T)

,NOME,FAZ,TAL,tipocorte,variedad,madurada,producto,dosismad,semsmad,edad,cortes,me,vejez,sacarosa,mes,periodo,TCH,lluvias,grupo_tenencia,pct_diatrea
0,AMAIME SILCA,81291,40,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,8.3,12.3,4,12.7,2.4,14.0,12,202012,112,137,3,6.2
1,AMAIME SILCA,81291,41,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,6.3,11.2,2,7.8,2.3,13.0,3,201903,157,0,3,3.5
2,AMAIME SILCA,81291,41,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.6,7.9,12.2,3,8.8,1.8,13.3,3,202003,167,68,3,4.3
3,AMAIME SILCA,81291,43,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,6.6,13.1,1,6.1,2.5,13.4,3,201903,156,0,3,3.5
4,AMAIME SILCA,81291,43,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.6,8.1,12.2,2,7.9,2.1,14.0,3,202003,151,68,3,4.3


<class 'pandas.DataFrame'>
RangeIndex: 2187 entries, 0 to 2186
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NOME            2187 non-null   str    
 1   FAZ             2187 non-null   int64  
 2   TAL             2187 non-null   object 
 3   tipocorte       2187 non-null   str    
 4   variedad        2187 non-null   str    
 5   madurada        2187 non-null   str    
 6   producto        2187 non-null   str    
 7   dosismad        2187 non-null   float64
 8   semsmad         2187 non-null   float64
 9   edad            2187 non-null   float64
 10  cortes          2187 non-null   int64  
 11  me              2187 non-null   float64
 12  vejez           2187 non-null   float64
 13  sacarosa        2187 non-null   float64
 14  mes             2187 non-null   int64  
 15  periodo         2187 non-null   int64  
 16  TCH             2187 non-null   int64  
 17  lluvias         2187 non-null   int64  
 18 

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
NOME,2187,285,SAN MIGUEL CARVAJAL,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
FAZ,2187.0,NaN,NaN,NaN,80588.332876,572.818299,80100.0,80222.0,80396.0,80660.0,82519.0
TAL,2187.0,273.0,1.0,258.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipocorte,2187,1,Mecanizado Verde,2187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variedad,2187,1,CC01-1940,2187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
madurada,2187,1,SI,2187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
producto,2187,1,BONUS 250 EC REGULADOR FISIOLÓGICO,2187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dosismad,2187.0,NaN,NaN,NaN,0.993278,0.309096,0.0,0.8,1.0,1.2,9.0
semsmad,2187.0,NaN,NaN,NaN,9.164838,3.441579,-1.6,7.1,8.7,10.6,45.0
edad,2187.0,NaN,NaN,NaN,12.766118,1.117866,10.3,12.0,12.5,13.3,21.1
